In [ ]:
import wandb
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random

In [ ]:
WANDB_API_KEY = "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"
WANDB_TEAM = "24f1001529--dl-genai-project" 
WANDB_PROJECT = "surface-anomaly-segmentation"
wandb.login(key=WANDB_API_KEY)

In [ ]:
BASE_DIR = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public") 
TRAIN_CSV = BASE_DIR / "train.csv"
SAMPLE_CSV = BASE_DIR / "sample_submission.csv"

TRAIN_IMG_DIR = BASE_DIR / "train"
TEST_IMG_DIR = BASE_DIR / "test"

train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_CSV)
print("Training annotation shape:", train_df.shape)
print("Test submission template shape:", sample_df.shape)
print("Training columns:", train_df.columns.tolist())
print("Submission columns:", sample_df.columns.tolist())

# Exploratory Data Analysis

In [ ]:
print(train_df.head())
print(sample_df.head())

In [ ]:
print("\nData types:\n", train_df.dtypes)
print("\nMissing values:\n", train_df.isnull().sum())
print("\nDuplicate rows:", train_df.duplicated().sum())
print("Duplicate image/category pairs:",
      train_df.duplicated(["image_id", "defect_type"]).sum())

print("\nUnique image IDs:", train_df["image_id"].nunique())
print("Total annotation rows:", len(train_df))

In [ ]:
train_df["mask_rle"] = train_df["mask_rle"].fillna("").astype(str).str.strip()
train_df["is_empty"] = train_df["mask_rle"].eq("")

print("\nEmpty masks:", train_df["is_empty"].sum())
print("Non-empty masks:", (~train_df["is_empty"]).sum())

In [ ]:
category_summary = train_df.groupby("defect_type").agg(
    annotation_rows=("image_id", "size"),
    unique_images=("image_id", "nunique"),
    empty_masks=("is_empty", "sum")
)
category_summary["nonempty_masks"] = (
    category_summary["annotation_rows"] - category_summary["empty_masks"]
)
category_summary["empty_percent"] = (
    100 * category_summary["empty_masks"] / category_summary["annotation_rows"]
)

print("\nCategory summary:")
display(category_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_df["defect_type"].value_counts().plot(
    kind="bar", ax=axes[0], title="Annotation Rows per Defect Type"
)
category_summary[["empty_masks", "nonempty_masks"]].plot(
    kind="bar", ax=axes[1], title="Empty vs Non-empty Masks"
)
axes[0].set_ylabel("Count")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
train_images = sorted(
    p for p in TRAIN_IMG_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
test_images = sorted(
    p for p in TEST_IMG_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

print("Training image files:", len(train_images))
print("Annotated image IDs:", train_df["image_id"].nunique())
print("Test image files:", len(test_images))
print("Template image IDs:", sample_df["image_id"].nunique())

image_path_map = {p.stem: p for p in train_images}
test_image_ids = {p.stem for p in test_images}

annotated_ids = set(train_df["image_id"].astype(str))
print("Annotated IDs without matching image files:",
      len(annotated_ids - set(image_path_map)))
print("Test image files without template IDs:",
      len(test_image_ids - set(sample_df["image_id"].astype(str))))

In [ ]:
image_info = []
for path in train_images:
    try:
        with Image.open(path) as img:
            image_info.append({
                "filename": path.name,
                "width": img.width,
                "height": img.height,
                "mode": img.mode
            })
    except Exception as e:
        print("Could not read:", path.name, e)

image_info_df = pd.DataFrame(image_info)

print("\nImage dimensions:")
display(
    image_info_df.groupby(["width", "height"]).size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("\nImage modes:")
display(image_info_df["mode"].value_counts())

In [ ]:
# RLE DECODER - COLUMN-MAJOR ORDER
def decode_rle(mask_rle, height, width):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(mask_rle) or str(mask_rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(mask_rle).split(), dtype=np.int64)

    if len(values) % 2 != 0:
        raise ValueError("RLE must contain start-length pairs")

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        if start < 0 or length < 0 or start + length > len(mask):
            raise ValueError("RLE values exceed image dimensions")
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")

In [ ]:
# MASK AREA AND COVERAGE STATISTICS
mask_stats = []

for _, row in train_df.iterrows():
    path = image_path_map.get(str(row["image_id"]))
    if path is None:
        continue

    try:
        with Image.open(path) as img:
            width, height = img.size

        mask = decode_rle(row["mask_rle"], height, width)
        pixels = int(mask.sum())

        mask_stats.append({
            "image_id": row["image_id"],
            "defect_type": row["defect_type"],
            "mask_pixels": pixels,
            "coverage_percent": 100 * pixels / (height * width),
            "is_empty": pixels == 0
        })
    except Exception as e:
        print(f"Mask error for {row['image_id']}: {e}")

mask_stats_df = pd.DataFrame(mask_stats)

print("\nMask statistics:")
if not mask_stats_df.empty:
    display(
        mask_stats_df.groupby("defect_type")[
            ["mask_pixels", "coverage_percent"]
        ].describe()
    )

    positive_coverage = mask_stats_df.loc[
        mask_stats_df["mask_pixels"] > 0, "coverage_percent"
    ]

    plt.figure(figsize=(8, 4))
    plt.hist(positive_coverage, bins=30)
    plt.xlabel("Defect coverage (%)")
    plt.ylabel("Number of annotations")
    plt.title("Distribution of Non-empty Mask Coverage")
    plt.tight_layout()
    plt.show()

    for defect in sorted(mask_stats_df["defect_type"].unique()):
        values = mask_stats_df.loc[
            (mask_stats_df["defect_type"] == defect)
            & (mask_stats_df["mask_pixels"] > 0),
            "coverage_percent"
        ]
        plt.figure(figsize=(6, 3))
        plt.hist(values, bins=25)
        plt.xlabel("Defect coverage (%)")
        plt.ylabel("Count")
        plt.title(f"Mask Coverage: {defect}")
        plt.tight_layout()
        plt.show()

In [ ]:
# DISPLAY SAMPLE TRAINING IMAGES
n = min(8, len(train_images))
if n > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for ax, path in zip(axes, train_images[:n]):
        with Image.open(path) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(path.name)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle("Sample Training Images")
    plt.tight_layout()
    plt.show()

In [ ]:
# DISPLAY SAMPLE GROUND-TRUTH MASKS AND OVERLAYS
nonempty = train_df[~train_df["is_empty"]]
n = min(4, len(nonempty))

if n > 0:
    fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
    axes = np.atleast_2d(axes)

    shown = 0
    for _, row in nonempty.iterrows():
        path = image_path_map.get(str(row["image_id"]))
        if path is None:
            continue

        with Image.open(path) as img:
            image = np.array(img.convert("RGB"))
            width, height = img.size

        try:
            mask = decode_rle(row["mask_rle"], height, width)
        except ValueError as e:
            print("Skipping invalid mask:", row["image_id"], e)
            continue

        axes[shown, 0].imshow(image)
        axes[shown, 0].set_title(f"Image: {row['image_id']}")
        axes[shown, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[shown, 1].set_title(f"Mask: {row['defect_type']}")
        axes[shown, 2].imshow(image)
        axes[shown, 2].imshow(mask, cmap="Reds", alpha=0.5)
        axes[shown, 2].set_title("Overlay")

        for ax in axes[shown]:
            ax.axis("off")

        shown += 1
        if shown == n:
            break

    for i in range(shown, n):
        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Columns:", sample_df.columns.tolist())
print("Rows:", len(sample_df))
print("Unique image IDs:", sample_df["image_id"].nunique())
print("\nRows per image:")
display(sample_df.groupby("image_id").size().value_counts().sort_index())
print("\nDefect category counts:")
display(sample_df["defect_type"].value_counts())

In [ ]:

# FURTHER EDA: SPATIAL PATTERNS & CO-OCCURRENCE

import random

DEFECTS = sorted(train_df["defect_type"].unique())

# 1. Which defect categories occur together?
presence = (
    train_df.assign(
        has_defect=train_df["mask_rle"].fillna("").astype(str).str.strip().ne("")
    )
    .pivot(index="image_id", columns="defect_type", values="has_defect")
    .fillna(False)
    .astype(int)
)

print("Defect presence per category:")
display(presence.sum().to_frame("positive_images"))

print("Number of images containing each pair of defects:")
display(presence.T.dot(presence))

# 2. Spatial heatmaps of positive mask pixels
# Each heatmap shows where defects tend to occur across the image.
spatial_maps = {
    defect: np.zeros((128, 800), dtype=np.float64)
    for defect in DEFECTS
}
positive_counts = {defect: 0 for defect in DEFECTS}

for _, row in train_df.iterrows():
    rle = row["mask_rle"]
    if pd.isna(rle) or not str(rle).strip():
        continue

    path = image_path_map.get(str(row["image_id"]))
    if path is None:
        continue

    mask = decode_rle(rle, int(row["height"]), int(row["width"]))

    if mask.shape != (128, 800):
        mask = np.array(
            Image.fromarray(mask).resize((800, 128), resample=Image.Resampling.NEAREST)
        )

    spatial_maps[row["defect_type"]] += mask
    positive_counts[row["defect_type"]] += 1

fig, axes = plt.subplots(2, 2, figsize=(16, 6))

for ax, defect in zip(axes.flat, DEFECTS):
    if positive_counts[defect] > 0:
        heatmap = spatial_maps[defect] / positive_counts[defect]
        im = ax.imshow(heatmap, cmap="hot", aspect="auto")
        ax.set_title(f"{defect}: Average mask location")
        ax.set_xlabel("Image X")
        ax.set_ylabel("Image Y")
        plt.colorbar(im, ax=ax, fraction=0.03)

plt.tight_layout()
plt.show()

# 3. Defect bounding boxes and approximate connected-region counts
try:
    from scipy.ndimage import label

    region_stats = []

    for _, row in train_df[train_df["mask_rle"].fillna("").astype(str).str.strip().ne("")].iterrows():
        path = image_path_map.get(str(row["image_id"]))
        if path is None:
            continue

        mask = decode_rle(row["mask_rle"], int(row["height"]), int(row["width"]))
        ys, xs = np.where(mask > 0)

        if len(xs) == 0:
            continue

        _, n_components = label(mask)

        region_stats.append({
            "defect_type": row["defect_type"],
            "area_pixels": int(mask.sum()),
            "bbox_width": int(xs.max() - xs.min() + 1),
            "bbox_height": int(ys.max() - ys.min() + 1),
            "connected_regions": int(n_components)
        })

    region_df = pd.DataFrame(region_stats)

    print("Bounding-box and connected-region statistics:")
    display(region_df.groupby("defect_type").agg(
        area_median=("area_pixels", "median"),
        bbox_width_median=("bbox_width", "median"),
        bbox_height_median=("bbox_height", "median"),
        regions_median=("connected_regions", "median"),
        regions_max=("connected_regions", "max")
    ))

    for defect in DEFECTS:
        vals = region_df.loc[
            region_df["defect_type"] == defect, "area_pixels"
        ]
        plt.figure(figsize=(6, 3))
        plt.hist(np.log10(vals.clip(lower=1)), bins=30)
        plt.title(f"{defect}: Log10 mask area")
        plt.xlabel("Log10 foreground pixels")
        plt.ylabel("Count")
        plt.show()

except ImportError:
    print("Install scipy to calculate connected-region statistics.")

In [ ]:

# IMAGE APPEARANCE
rng = np.random.default_rng(42)
sample_ids = rng.choice(
    train_df["image_id"].unique(),
    size=min(500, train_df["image_id"].nunique()),
    replace=False
)

appearance = []

for image_id in sample_ids:
    path = image_path_map.get(str(image_id))
    if path is None:
        continue

    with Image.open(path) as img:
        gray = np.asarray(img.convert("L"), dtype=np.float32)

    row = {
        "image_id": image_id,
        "mean_brightness": float(gray.mean()),
        "contrast_std": float(gray.std())
    }

    for defect in DEFECTS:
        mask_row = train_df[
            (train_df["image_id"] == image_id)
            & (train_df["defect_type"] == defect)
        ]

        row[defect] = (
            len(mask_row) > 0
            and mask_row["mask_rle"].fillna("").astype(str).str.strip().ne("").any()
        )

    appearance.append(row)

appearance_df = pd.DataFrame(appearance)

for defect in DEFECTS:
    appearance_df[defect] = appearance_df[defect].map({
        True: "Defect present",
        False: "Mask empty"
    })

    print(f"\n{defect}: brightness and contrast")
    display(
        appearance_df.groupby(defect)[
            ["mean_brightness", "contrast_std"]
        ].agg(["count", "mean", "median", "std"])
    )

    appearance_df.boxplot(
        column="contrast_std",
        by=defect,
        figsize=(7, 4)
    )
    plt.title(f"Image contrast by {defect}")
    plt.suptitle("")
    plt.ylabel("Grayscale standard deviation")
    plt.show()


In [ ]:

# DEFECT DIFFICULTY 
for defect in sorted(train_df["defect_type"].unique()):
    stats = mask_stats_df[
        (mask_stats_df["defect_type"] == defect)
        & (mask_stats_df["mask_pixels"] > 0)
    ].sort_values("mask_pixels")

    if stats.empty:
        continue

    # Smallest, median-sized, and largest positive masks
    selected = stats.iloc[
        sorted(set([0, len(stats) // 2, len(stats) - 1]))
    ]

    fig, axes = plt.subplots(len(selected), 3, figsize=(12, 4 * len(selected)))
    axes = np.atleast_2d(axes)

    for i, (_, stat) in enumerate(selected.iterrows()):
        row = train_df[
            (train_df["image_id"] == stat["image_id"])
            & (train_df["defect_type"] == defect)
        ].iloc[0]

        path = image_path_map.get(str(row["image_id"]))
        if path is None:
            continue

        with Image.open(path) as img:
            image = np.array(img.convert("RGB"))
            width, height = img.size

        mask = decode_rle(row["mask_rle"], height, width)

        axes[i, 0].imshow(image)
        axes[i, 0].set_title("Original")

        axes[i, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[i, 1].set_title(f"Mask: {int(mask.sum())} pixels")

        axes[i, 2].imshow(image)
        axes[i, 2].imshow(mask, cmap="Reds", alpha=0.6)
        axes[i, 2].set_title(
            f"{defect} | coverage {100 * mask.mean():.3f}%"
        )

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle(f"{defect}: Small, median, and large defects")
    plt.tight_layout()
    plt.show()

# Verify that positive-mask area is consistent with RLE run lengths
def rle_pixel_count(rle):
    if pd.isna(rle) or not str(rle).strip():
        return 0
    values = np.asarray(str(rle).split(), dtype=np.int64)
    return int(values[1::2].sum())

train_df["rle_pixel_count"] = train_df["mask_rle"].apply(rle_pixel_count)

print("RLE area consistency check:")
display(
    train_df.groupby("defect_type").apply(
        lambda g: pd.Series({
            "positive_masks": int((g["rle_pixel_count"] > 0).sum()),
            "minimum_positive_area": int(
                g.loc[g["rle_pixel_count"] > 0, "rle_pixel_count"].min()
            ),
            "median_positive_area": float(
                g.loc[g["rle_pixel_count"] > 0, "rle_pixel_count"].median()
            ),
            "maximum_area": int(g["rle_pixel_count"].max())
        }),
        include_groups=False
    )
)


# Preprocessing

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import torch

# Ensure IDs have a consistent representation
train_df["image_id"] = train_df["image_id"].astype(str)
image_path_map = {str(k): v for k, v in image_path_map.items()}

# Preserve a consistent channel order for the four defect categories
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]

# Native image dimensions
IMAGE_HEIGHT = 128
IMAGE_WIDTH = 800
NUM_DEFECTS = len(DEFECTS)

# RGB normalization values for a future ImageNet-pretrained encoder
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406], dtype=torch.float32
).view(3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225], dtype=torch.float32
).view(3, 1, 1)

print("Defect categories:", DEFECTS)
print("Image dimensions:", IMAGE_HEIGHT, "x", IMAGE_WIDTH)
print("Number of mask channels:", NUM_DEFECTS)
print("Training images:", len(image_path_map))
print("Training annotation rows:", len(train_df))
print("PyTorch device:", "CUDA" if torch.cuda.is_available() else "CPU")

In [ ]:
# Check the expected categories
assert set(train_df["defect_type"].unique()) == set(DEFECTS), (
    "Unexpected or missing defect categories."
)

# Every image should have exactly one row per defect category
category_counts = train_df.groupby("image_id")["defect_type"].nunique()

assert (category_counts == NUM_DEFECTS).all(), (
    "Some images do not have all four category annotations."
)

assert not train_df.duplicated(
    subset=["image_id", "defect_type"]
).any(), "Duplicate image/category annotations found."

annotation_ids = set(train_df["image_id"])
image_ids = set(image_path_map)

print("Annotation IDs:", len(annotation_ids))
print("Image IDs:", len(image_ids))
print("Annotations without images:", len(annotation_ids - image_ids))
print("Images without annotations:", len(image_ids - annotation_ids))

assert annotation_ids == image_ids, (
    "Image IDs and annotation IDs do not match."
)

# Verify actual dimensions for all training images
dimension_errors = []

for image_id, image_path in image_path_map.items():
    with Image.open(image_path) as img:
        actual_width, actual_height = img.size

    rows = train_df[train_df["image_id"] == image_id]
    expected_width = int(rows["width"].iloc[0])
    expected_height = int(rows["height"].iloc[0])

    if (
        actual_width != expected_width
        or actual_height != expected_height
    ):
        dimension_errors.append(
            (image_id, actual_width, actual_height,
             expected_width, expected_height)
        )

print("Dimension mismatches:", len(dimension_errors))

if dimension_errors:
    print("First mismatches:", dimension_errors[:10])
    raise ValueError("Resolve image dimension mismatches before proceeding.")

In [ ]:
# Use the existing EDA preprocessing of mask_rle if already performed.
train_df["mask_rle"] = (
    train_df["mask_rle"].fillna("").astype(str).str.strip()
)

# Map each image ID to its category annotations
annotation_lookup = {
    (str(row.image_id), str(row.defect_type)): row.mask_rle
    for row in train_df[
        ["image_id", "defect_type", "mask_rle"]
    ].itertuples(index=False)
}

print("Annotation lookup entries:", len(annotation_lookup))

assert len(annotation_lookup) == len(train_df), (
    "Lookup size differs from annotation row count."
)

In [ ]:
def prepare_sample(image_id, imagenet_normalize=False):
    """
    Prepare one image and its four category masks.

    Returns:
        image_tensor: [3, 128, 800], float32
        mask_tensor:  [4, 128, 800], float32

    Image values are in [0, 1] by default.
    Masks contain only 0 and 1.
    """

    image_id = str(image_id)

    if image_id not in image_path_map:
        raise KeyError(f"Image not found: {image_id}")

    image_path = image_path_map[image_id]

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        image_array = np.asarray(img, dtype=np.uint8).copy()

    height, width, channels = image_array.shape

    if channels != 3:
        raise ValueError(f"Expected RGB image, got {channels} channels.")

    if (height, width) != (IMAGE_HEIGHT, IMAGE_WIDTH):
        raise ValueError(
            f"Unexpected dimensions for {image_id}: "
            f"{height} x {width}"
        )

    # Convert HWC uint8 RGB into CHW float32 in [0, 1]
    image_tensor = torch.from_numpy(
        image_array.transpose(2, 0, 1)
    ).float() / 255.0

    mask_list = []

    for defect in DEFECTS:
        rle = annotation_lookup[(image_id, defect)]

        mask_array = decode_rle(
            rle,
            height=height,
            width=width
        )

        # Enforce binary mask representation
        mask_array = (mask_array > 0).astype(np.float32)

        if mask_array.shape != (height, width):
            raise ValueError(
                f"Incorrect mask shape for {image_id}, {defect}"
            )

        mask_list.append(mask_array)

    # [4, H, W]
    mask_tensor = torch.from_numpy(
        np.stack(mask_list, axis=0)
    ).float()

    if imagenet_normalize:
        image_tensor = (
            image_tensor - IMAGENET_MEAN
        ) / IMAGENET_STD

    return image_tensor, mask_tensor

In [ ]:
# Select a reproducible example
example_id = sorted(image_path_map.keys())[0]

image_tensor, mask_tensor = prepare_sample(example_id)

print("Example image ID:", example_id)
print("Image tensor shape:", tuple(image_tensor.shape))
print("Image tensor dtype:", image_tensor.dtype)
print("Image range:", image_tensor.min().item(),
      image_tensor.max().item())

print("Mask tensor shape:", tuple(mask_tensor.shape))
print("Mask tensor dtype:", mask_tensor.dtype)
print("Mask values:", torch.unique(mask_tensor))

for i, defect in enumerate(DEFECTS):
    pixels = int(mask_tensor[i].sum().item())
    print(f"{defect}: {pixels:,} foreground pixels")

assert image_tensor.shape == (3, IMAGE_HEIGHT, IMAGE_WIDTH)
assert mask_tensor.shape == (
    NUM_DEFECTS, IMAGE_HEIGHT, IMAGE_WIDTH
)
assert torch.isfinite(image_tensor).all()
assert torch.isfinite(mask_tensor).all()
assert set(torch.unique(mask_tensor).tolist()).issubset({0.0, 1.0})

In [ ]:
image_tensor, mask_tensor = prepare_sample(example_id)

# Convert image tensor back to HWC for visualization
image_display = image_tensor.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(
    2, 3, figsize=(18, 8), constrained_layout=True
)

axes[0, 0].imshow(image_display)
axes[0, 0].set_title(f"Original image: {example_id}")
axes[0, 0].axis("off")

for i, defect in enumerate(DEFECTS):
    ax = axes.flat[i + 1]
    ax.imshow(mask_tensor[i].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"{defect} | {int(mask_tensor[i].sum().item()):,} pixels"
    )
    ax.axis("off")

axes[1, 2].imshow(image_display)

# Overlay all positive masks using different colors for inspection
overlay_colors = [
    np.array([1.0, 0.0, 0.0]),  # defect_a
    np.array([0.0, 1.0, 0.0]),  # defect_b
    np.array([0.0, 0.0, 1.0]),  # defect_c
    np.array([1.0, 1.0, 0.0]),  # defect_d
]

overlay = image_display.copy()

for i, color in enumerate(overlay_colors):
    mask = mask_tensor[i].numpy().astype(bool)
    overlay[mask] = (
        0.5 * overlay[mask] + 0.5 * color
    )

axes[1, 2].imshow(overlay)
axes[1, 2].set_title("All-category mask overlay")
axes[1, 2].axis("off")

plt.show()

In [ ]:
rng = np.random.default_rng(42)

all_ids = np.array(sorted(image_path_map.keys()))
n_check = min(50, len(all_ids))

check_ids = rng.choice(
    all_ids,
    size=n_check,
    replace=False
)

checked_positive_pairs = 0
checked_empty_pairs = 0

for image_id in check_ids:
    image_tensor, mask_tensor = prepare_sample(image_id)

    assert image_tensor.shape == (3, IMAGE_HEIGHT, IMAGE_WIDTH)
    assert mask_tensor.shape == (
        NUM_DEFECTS, IMAGE_HEIGHT, IMAGE_WIDTH
    )

    assert torch.isfinite(image_tensor).all()
    assert torch.isfinite(mask_tensor).all()

    for channel in range(NUM_DEFECTS):
        pixel_count = int(mask_tensor[channel].sum().item())

        if pixel_count > 0:
            checked_positive_pairs += 1
        else:
            checked_empty_pairs += 1

print("Images checked:", n_check)
print("Positive image/category pairs:", checked_positive_pairs)
print("Empty image/category pairs:", checked_empty_pairs)

# xxx

In [ ]:
# CELL A — imports, reproducibility, and configuration
import os, time, copy, random, itertools, math, json, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import (
    deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
)

import wandb

SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 8                 # Increase after validating the pipeline.
PATIENCE = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
USE_PRETRAINED = True
WANDB_PROJECT = "industrial-defect-segmentation"
WANDB_ENTITY = None        # Set your team/entity string if needed.
RUN_ALL_EXPERIMENTS = True
MAX_EXPERIMENTS = None     # Set e.g. 4 for a quick smoke test.
PIN_MEMORY = torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Ensure consistent column order and clean annotations.
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
train_df["image_id"] = train_df["image_id"].astype(str)
train_df["mask_rle"] = train_df["mask_rle"].fillna("").astype(str).str.strip()
image_path_map = {str(k): Path(v) for k, v in image_path_map.items()}

assert set(train_df["defect_type"].unique()) == set(DEFECTS)
assert "decode_rle" in globals(), "Run the existing RLE decoder cell first."
assert len(image_path_map) > 0

In [ ]:
# CELL B — image-level train / validation / test split
# Split by image ID, never by annotation row, to prevent image leakage.

from sklearn.model_selection import train_test_split

image_ids = np.array(sorted(set(train_df["image_id"]) & set(image_path_map)))
presence_table = (
    train_df.assign(
        present=train_df["mask_rle"].str.strip().ne("")
    )
    .pivot_table(
        index="image_id", columns="defect_type", values="present",
        aggfunc="max", fill_value=False
    )
    .reindex(index=image_ids, columns=DEFECTS, fill_value=False)
    .astype(int)
)

# Use a deterministic random split. Multilabel stratification is not assumed
# available; report each split's per-category positive counts to reveal drift.
train_ids, temp_ids = train_test_split(
    image_ids, test_size=0.20, random_state=SEED, shuffle=True
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, random_state=SEED, shuffle=True
)

split_ids = {
    "train": list(train_ids),
    "val": list(val_ids),
    "test": list(test_ids),
}

def show_split_stats():
    rows = []
    for split_name, ids in split_ids.items():
        p = presence_table.reindex(ids, fill_value=0)
        row = {"split": split_name, "images": len(ids)}
        row.update({f"{d}_positive_images": int(p[d].sum()) for d in DEFECTS})
        rows.append(row)
    return pd.DataFrame(rows)

display(show_split_stats())
assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))

In [ ]:
# CELL C — dataset, synchronized augmentation, and mask construction
# Image/mask tensors remain [3,H,W] and [4,H,W]. No resizing is performed.

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def transform_pair(image, mask, transform_name):
    if transform_name == "none":
        return image, mask
    if transform_name == "horizontal_flip":
        if torch.rand(()) < 0.5:
            image, mask = TF.hflip(image), TF.hflip(mask)
        return image, mask
    if transform_name == "brightness_contrast":
        image = TF.adjust_brightness(image, float(torch.empty(()).uniform_(0.90, 1.10)))
        image = TF.adjust_contrast(image, float(torch.empty(()).uniform_(0.90, 1.10)))
        return image.clamp(0, 1), mask
    if transform_name == "small_translation":
        h, w = image.shape[-2:]
        dx = int(torch.randint(-max(1, int(w * 0.01)), max(1, int(w * 0.01)) + 1, ()).item())
        dy = int(torch.randint(-max(1, int(h * 0.03)), max(1, int(h * 0.03)) + 1, ()).item())
        image = TF.affine(
            image, angle=0, translate=[dx, dy], scale=1.0, shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR, fill=0
        )
        mask = TF.affine(
            mask, angle=0, translate=[dx, dy], scale=1.0, shear=[0.0, 0.0],
            interpolation=InterpolationMode.NEAREST, fill=0
        )
        return image, (mask > 0.5).float()
    raise ValueError(f"Unknown transform: {transform_name}")

class DefectDataset(Dataset):
    def __init__(self, ids, annotations, image_paths, training=False,
                 transform_name="none", feature_mode="rgb_only"):
        self.ids = [str(x) for x in ids]
        self.annotations = annotations
        self.image_paths = image_paths
        self.training = training
        self.transform_name = transform_name if training else "none"
        self.feature_mode = feature_mode

        sub = annotations[annotations["image_id"].isin(self.ids)]
        self.rle_lookup = {
            (str(r.image_id), str(r.defect_type)): str(r.mask_rle or "")
            for r in sub[["image_id", "defect_type", "mask_rle"]].itertuples(index=False)
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        path = self.image_paths[image_id]
        with Image.open(path) as im:
            rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)
        image = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float() / 255.0
        h, w = image.shape[-2:]

        masks = []
        for defect in DEFECTS:
            rle = self.rle_lookup.get((image_id, defect), "")
            decoded = decode_rle(rle, h, w)
            masks.append(torch.from_numpy(np.asarray(decoded, dtype=np.float32)))
        mask = torch.stack(masks, dim=0)

        image, mask = transform_pair(image, mask, self.transform_name)

        # Correct normalization for ImageNet-pretrained encoder.
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        return image, mask, image_id

def make_category_sampler(ids, annotations):
    # Moderate inverse-frequency weighting based on image-level category presence.
    p = presence_table.reindex(ids, fill_value=0)
    counts = p.sum(axis=0).clip(lower=1)
    rarity = (len(p) / counts) ** 0.5
    rarity = rarity / rarity.mean()
    weights = np.ones(len(ids), dtype=np.float64)
    for d in DEFECTS:
        weights += p[d].to_numpy(dtype=float) * max(0.0, float(rarity[d]) - 1.0)
    return WeightedRandomSampler(
        torch.as_tensor(np.maximum(weights, 1e-6), dtype=torch.double),
        num_samples=len(ids), replacement=True
    )

In [ ]:
# CELL D — model, losses, and competition-oriented validation metrics

def build_model(feature_mode="rgb_only", pretrained=True):
    weights = DeepLabV3_ResNet50_Weights.DEFAULT if pretrained else None

    model = deeplabv3_resnet50(
        weights=weights,
        aux_loss=True if pretrained else False
    )

    # Replace the main segmentation head with four independent outputs.
    old = model.classifier[-1]
    model.classifier[-1] = nn.Conv2d(
        old.in_channels,
        len(DEFECTS),
        kernel_size=1
    )

    # The auxiliary head is not used for the training loss.
    # Keep it instantiated to satisfy the pretrained model configuration.

    if feature_mode == "gradient_features":
        old_conv = model.backbone.conv1

        new_conv = nn.Conv2d(
            6,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            mean_kernel = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight[:, 3:] = mean_kernel.repeat(1, 3, 1, 1) / 3.0

        model.backbone.conv1 = new_conv

    elif feature_mode != "rgb_only":
        raise ValueError(f"Unsupported feature mode: {feature_mode}")

    return model

def add_gradient_channels(images):
    # Input is already ImageNet-normalized. Undo normalization before deriving
    # gradients, then append gradients to the normalized RGB channels.
    mean = IMAGENET_MEAN.to(images.device)
    std = IMAGENET_STD.to(images.device)
    rgb = (images * std + mean).clamp(0, 1)
    gray = 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]
    sx = torch.tensor(
        [[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
        device=images.device, dtype=images.dtype
    ).view(1, 1, 3, 3) / 8
    sy = sx.transpose(-1, -2)
    gx = F.conv2d(gray, sx, padding=1)
    gy = F.conv2d(gray, sy, padding=1)
    mag = torch.sqrt(gx.square() + gy.square() + 1e-8)
    # Scale gradient channels conservatively.
    return torch.cat([images, mag, gx, gy], dim=1)

def bce_dice_loss(logits, targets, dice_weight=0.5, eps=1e-6):
    bce = F.binary_cross_entropy_with_logits(logits, targets.float())
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3)
    inter = (probs * targets).sum(dims)
    denom = probs.sum(dims) + targets.sum(dims)
    dice = ((2 * inter + eps) / (denom + eps)).mean()
    return (1 - dice_weight) * bce + dice_weight * (1 - dice)

def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    targets = targets.float()
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)
    pt = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return (alpha_t * (1 - pt).pow(gamma) * bce).mean()

def choose_loss(name, logits, targets):
    if name == "bce":
        return F.binary_cross_entropy_with_logits(logits, targets.float())
    if name == "bce_dice":
        return bce_dice_loss(logits, targets)
    if name == "focal":
        return focal_loss(logits, targets)
    raise ValueError(name)

@torch.no_grad()
def metric_counts_from_batch(logits, targets, threshold=0.5):
    pred = torch.sigmoid(logits) >= threshold
    true = targets >= 0.5
    dims = (0, 2, 3)
    tp = (pred & true).sum(dims).double()
    fp = (pred & ~true).sum(dims).double()
    fn = (~pred & true).sum(dims).double()
    gt_positive_pairs = true.flatten(2).any(dim=2).sum(dim=0).double()
    gt_empty_pairs = (~true.flatten(2).any(dim=2)).sum(dim=0).double()
    correct_empty_pairs = ((~pred.flatten(2).any(dim=2)) & (~true.flatten(2).any(dim=2))).sum(dim=0).double()
    pred_on_empty_pairs = (pred.flatten(2).any(dim=2) & (~true.flatten(2).any(dim=2))).sum(dim=0).double()
    return tp, fp, fn, gt_positive_pairs, gt_empty_pairs, correct_empty_pairs, pred_on_empty_pairs

def metrics_from_counts(counts, eps=1e-8):
    tp, fp, fn, pos_pairs, empty_pairs, correct_empty, false_positive_pairs = counts
    # Foreground Dice pooled by category over all positive pixels.
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    # Transparent proxy definitions pending official evaluator details:
    # FalsePositiveScore = 1 - fraction of empty image/category pairs with any prediction.
    # EmptyAccuracy = fraction of empty image/category pairs predicted empty.
    empty_denom = empty_pairs.clamp(min=1)
    fp_score = 1.0 - false_positive_pairs / empty_denom
    empty_acc = correct_empty / empty_denom
    # Do not let categories with no positive pixels create a perfect Dice score:
    # exclude them from foreground Dice and report their absence through empty metrics.
    dice = torch.where(pos_pairs > 0, dice, torch.nan)
    foreground_dice = torch.nanmean(dice).item()
    false_positive_score = fp_score.mean().item()
    empty_accuracy = empty_acc.mean().item()
    score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "foreground_dice": foreground_dice,
        "false_positive_score_proxy": false_positive_score,
        "empty_accuracy": empty_accuracy,
        "competition_score_proxy": score,
        "per_category_dice": {
            d: (None if torch.isnan(dice[i]) else float(dice[i]))
            for i, d in enumerate(DEFECTS)
        }
    }

In [ ]:
# CELL E — train/validation/test loops, timing, and resource logging

def prepare_model_input(images, feature_mode):
    return add_gradient_channels(images) if feature_mode == "gradient_features" else images

def run_epoch(model, loader, optimizer=None, loss_name="bce_dice",
              feature_mode="rgb_only", scaler=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, n_seen = 0.0, 0
    counts = None
    start = time.perf_counter()

    for images, masks, _ in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        model_input = prepare_model_input(images, feature_mode)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(model_input)["out"]
            # Upsample logits to native mask size if architecture output differs.
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(
                    logits, size=masks.shape[-2:], mode="bilinear",
                    align_corners=False
                )
            loss = choose_loss(loss_name, logits, masks)

            if training:
                loss.backward()
                optimizer.step()

        batch_size = images.shape[0]
        loss_sum += loss.item() * batch_size
        n_seen += batch_size

        batch_counts = metric_counts_from_batch(logits.detach(), masks)
        if counts is None:
            counts = [x.clone() for x in batch_counts]
        else:
            counts = [a + b for a, b in zip(counts, batch_counts)]

    elapsed = time.perf_counter() - start
    result = {"loss": loss_sum / max(n_seen, 1), "seconds": elapsed}
    result.update(metrics_from_counts(counts))
    result["images_per_second"] = n_seen / max(elapsed, 1e-8)
    return result

def gpu_memory_gb():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.max_memory_allocated() / (1024 ** 3)

def make_loaders(config):
    train_ds = DefectDataset(
        split_ids["train"], train_df, image_path_map, training=True,
        transform_name=config["transform"], feature_mode=config["feature"]
    )
    val_ds = DefectDataset(
        split_ids["val"], train_df, image_path_map, training=False,
        feature_mode=config["feature"]
    )
    test_ds = DefectDataset(
        split_ids["test"], train_df, image_path_map, training=False,
        feature_mode=config["feature"]
    )

    sampler = make_category_sampler(split_ids["train"], train_df) if config["sampler"] == "category" else None
    common = dict(
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0)
    )
    train_loader = DataLoader(train_ds, shuffle=(sampler is None), sampler=sampler, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)
    return train_loader, val_loader, test_loader

def train_experiment(config, experiment_index, total_experiments):
    run_name = (
        f"{experiment_index:03d}_{config['transform']}_{config['loss']}_"
        f"{config['sampler']}_{config['feature']}"
    )
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=run_name,
        config={
            **config, "seed": SEED, "batch_size": BATCH_SIZE,
            "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY, "threshold": THRESHOLD,
            "model": "DeepLabV3-ResNet50", "pretrained": USE_PRETRAINED,
            "train_images": len(split_ids["train"]),
            "val_images": len(split_ids["val"]),
            "test_images": len(split_ids["test"]),
            "score_metric_note": "False-positive proxy; replace with official exact evaluator if available."
        },
        reinit="finish_previous"
    )

    try:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        train_loader, val_loader, test_loader = make_loaders(config)
        model = build_model(config["feature"], pretrained=USE_PRETRAINED).to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )

        best_val = float("inf")
        best_state = None
        best_epoch = 0
        bad_epochs = 0
        history = []
        total_train_seconds = 0.0
        total_val_seconds = 0.0

        for epoch in range(1, EPOCHS + 1):
            epoch_start = time.perf_counter()
            train_metrics = run_epoch(
                model, train_loader, optimizer, config["loss"], config["feature"]
            )
            val_metrics = run_epoch(
                model, val_loader, None, config["loss"], config["feature"]
            )
            total_train_seconds += train_metrics["seconds"]
            total_val_seconds += val_metrics["seconds"]
            epoch_seconds = time.perf_counter() - epoch_start

            row = {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "val_loss": val_metrics["loss"],
                "train_score_proxy": train_metrics["competition_score_proxy"],
                "val_score_proxy": val_metrics["competition_score_proxy"],
                "train_foreground_dice": train_metrics["foreground_dice"],
                "val_foreground_dice": val_metrics["foreground_dice"],
                "val_false_positive_score_proxy": val_metrics["false_positive_score_proxy"],
                "val_empty_accuracy": val_metrics["empty_accuracy"],
                "train_seconds": train_metrics["seconds"],
                "validation_seconds": val_metrics["seconds"],
                "epoch_wall_seconds": epoch_seconds,
                "train_images_per_second": train_metrics["images_per_second"],
                "val_images_per_second": val_metrics["images_per_second"],
                "peak_gpu_memory_gb": gpu_memory_gb(),
            }
            history.append(row)
            wandb.log(row, step=epoch)

            print(
                f"[{experiment_index}/{total_experiments}] {run_name} | "
                f"epoch {epoch}/{EPOCHS} | "
                f"train loss {row['train_loss']:.4f} | "
                f"val loss {row['val_loss']:.4f} | "
                f"val score* {row['val_score_proxy']:.4f} | "
                f"{epoch_seconds:.1f}s",
                flush=True
            )

            # Use validation loss for early stopping, not test metrics.
            if val_metrics["loss"] < best_val:
                best_val = val_metrics["loss"]
                best_epoch = epoch
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= PATIENCE:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        # Test set is evaluated only after model selection is complete.
        test_start = time.perf_counter()
        test_metrics = run_epoch(
            model, test_loader, None, config["loss"], config["feature"]
        )
        test_seconds = time.perf_counter() - test_start

        summary = {
            "best_epoch": best_epoch,
            "best_val_loss": best_val,
            "final_train_loss": history[-1]["train_loss"],
            "final_val_loss": history[-1]["val_loss"],
            "test_loss": test_metrics["loss"],
            "test_foreground_dice": test_metrics["foreground_dice"],
            "test_false_positive_score_proxy": test_metrics["false_positive_score_proxy"],
            "test_empty_accuracy": test_metrics["empty_accuracy"],
            "test_score_proxy": test_metrics["competition_score_proxy"],
            "total_train_seconds": total_train_seconds,
            "total_validation_seconds": total_val_seconds,
            "test_evaluation_seconds": test_seconds,
            "total_run_seconds": sum(x["epoch_wall_seconds"] for x in history) + test_seconds,
            "peak_gpu_memory_gb": gpu_memory_gb(),
            "test_images_per_second": test_metrics["images_per_second"],
        }
        wandb.summary.update(summary)

        # Save history and best weights as run artifacts for reproducibility.
        history_path = f"{run_name}_history.csv"
        pd.DataFrame(history).to_csv(history_path, index=False)
        wandb.save(history_path)
        weights_path = f"{run_name}_best.pt"
        torch.save(best_state, weights_path)
        wandb.save(weights_path)

        return {"experiment": run_name, **config, **summary}
    finally:
        wandb.finish()
        if "model" in locals():
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
import os
import gc
import wandb
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader
from pathlib import Path

# ------------------------------------------------------------
# 1. Three experiments
# ------------------------------------------------------------
selected_experiments = [
    {
        "transform": "none",
        "loss": "bce",
        "sampler": "uniform",
        "feature": "rgb_only",
        "mask_representation": "binary_four_channel",
    },
    {
        "transform": "none",
        "loss": "bce_dice",
        "sampler": "uniform",
        "feature": "rgb_only",
        "mask_representation": "binary_four_channel",
    },
    {
        "transform": "brightness_contrast",
        "loss": "focal",
        "sampler": "category",
        "feature": "gradient_features",
        "mask_representation": "binary_four_channel",
    },
]

# Make W&B reinitialization compatible with the earlier function.
_original_wandb_init = wandb.init

def _compatible_wandb_init(*args, **kwargs):
    if kwargs.get("reinit") == "finish":
        kwargs["reinit"] = "finish_previous"
    return _original_wandb_init(*args, **kwargs)

wandb.init = _compatible_wandb_init

new_results = []
checkpoint_dir = Path("/kaggle/working")
os.makedirs(checkpoint_dir, exist_ok=True)

try:
    for idx, config in zip([11, 12, 13], selected_experiments):
        print("\n" + "=" * 70)
        print(f"Starting experiment {idx}: {config}")
        print("=" * 70)

        result = train_experiment(
            config=config,
            experiment_index=idx,
            total_experiments=13,
        )

        new_results.append(result)

        pd.DataFrame(new_results).to_csv(
            checkpoint_dir / "selected_experiments_results.csv",
            index=False,
        )
        print(f"Experiment {idx} completed and results saved.")

finally:
    wandb.init = _original_wandb_init
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. Compare experiments
# ------------------------------------------------------------
if not new_results:
    raise RuntimeError("No experiments completed; submission was not generated.")

selected_results_df = pd.DataFrame(new_results)
selected_results_df.to_csv(
    checkpoint_dir / "selected_experiments_results.csv",
    index=False,
)
display(
    selected_results_df.sort_values("best_val_loss").reset_index(drop=True)
)

# Select using validation loss, not test-set performance.
best_row = selected_results_df.loc[
    selected_results_df["best_val_loss"].idxmin()
]
best_run_name = best_row["experiment"]
best_config = {
    key: best_row[key]
    for key in ["transform", "loss", "sampler", "feature",
                "mask_representation"]
}
best_checkpoint = checkpoint_dir / f"{best_run_name}_best.pt"

print("\nSelected checkpoint:", best_checkpoint)
print("Selected configuration:", best_config)

if not best_checkpoint.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {best_checkpoint}. "
        "Check where train_experiment saves its model weights."
    )

# ------------------------------------------------------------
# 3. Find the test-image mapping and submission template
# ------------------------------------------------------------
# The existing notebook must already have loaded test image paths.
test_path_map = None
for variable_name in [
    "test_image_path_map",
    "test_image_paths",
    "test_path_map",
]:
    value = globals().get(variable_name)
    if isinstance(value, dict) and len(value) > 0:
        test_path_map = value
        print("Using test paths from:", variable_name)
        break

sample_sub = globals().get("sample_submission")

if test_path_map is None:
    raise RuntimeError(
        "Experiments completed, but I couldn't find a test-image mapping. "
        "Load/build a dictionary mapping each test image ID to its image path "
        "and assign it to test_image_path_map before running the submission section."
    )

if not isinstance(sample_sub, pd.DataFrame):
    raise RuntimeError(
        "Experiments completed, but sample_submission is not loaded as a DataFrame. "
        "Load the competition's sample submission CSV into sample_submission "
        "before running the submission section."
    )

# ------------------------------------------------------------
# 4. Load best model and generate masks
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = build_model(best_config["feature"], pretrained=False).to(device)

state_dict = torch.load(best_checkpoint, map_location="cpu", weights_only=True)
best_model.load_state_dict(state_dict)
best_model.eval()

test_ids = sorted(str(k) for k in test_path_map.keys())

# Use the competition's template to preserve its required row/column format.
submission = sample_sub.copy()

# Identify the columns used by the template.
print("Sample submission columns:", submission.columns.tolist())
print("Test-image count:", len(test_ids))

# This code supports the common long format:
# one row per image/category with columns image_id, defect_type, mask_rle.
required_cols = {"image_id", "defect_type", "mask_rle"}
if not required_cols.issubset(submission.columns):
    raise RuntimeError(
        "The sample-submission columns do not match the expected "
        "image_id / defect_type / mask_rle format. "
        "No submission was written. Check the displayed column names "
        "and adapt the encoding section to the official template."
    )

def encode_rle(mask):
    # Column-major encoding, matching the notebook's decoder.
    pixels = np.asarray(mask, dtype=np.uint8).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    changes = np.where(pixels[1:] != pixels[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes))

test_predictions = {}

with torch.no_grad():
    for start in range(0, len(test_ids), BATCH_SIZE):
        batch_ids = test_ids[start:start + BATCH_SIZE]
        batch_images = []

        for image_id in batch_ids:
            with Image.open(test_path_map[image_id]) as image:
                array = np.asarray(image.convert("RGB"), dtype=np.uint8).copy()

            tensor = torch.from_numpy(array).permute(2, 0, 1).float() / 255.0
            tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD
            batch_images.append(tensor)

        images = torch.stack(batch_images).to(device)

        if best_config["feature"] == "gradient_features":
            images = add_gradient_channels(images)

        logits = best_model(images)["out"]
        logits = F.interpolate(
            logits,
            size=(IMAGE_HEIGHT, IMAGE_WIDTH),
            mode="bilinear",
            align_corners=False,
        )
        masks = (torch.sigmoid(logits) >= THRESHOLD).cpu().numpy()

        for i, image_id in enumerate(batch_ids):
            test_predictions[image_id] = {
                defect: encode_rle(masks[i, j])
                for j, defect in enumerate(DEFECTS)
            }

        print(f"Predicted {min(start + len(batch_ids), len(test_ids))}/{len(test_ids)} images")

# Fill predictions by image/category; retain the exact template row order.
for row_idx, row in submission.iterrows():
    image_id = str(row["image_id"])
    defect = str(row["defect_type"])

    if image_id not in test_predictions:
        raise KeyError(f"No prediction generated for template image ID {image_id}")

    submission.at[row_idx, "mask_rle"] = test_predictions[image_id][defect]

submission_path = checkpoint_dir / "submission.csv"
submission.to_csv(submission_path, index=False)

print("\nSubmission saved to:", submission_path)
print("Submission rows:", len(submission))
display(submission.head())

del best_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
print(efgergcmergmhdskv)

In [ ]:
# CELL F — consolidated experiment registry and runner
# Only combinations that make sense for the competition are included:
# - Binary four-channel masks remain the target for every run.
# - Boundary/distance/multiscale targets are excluded from the core grid because
#   they need auxiliary heads/losses and do not directly match the score.
# - Handcrafted global features are excluded as primary inputs because they do
#   not localize pixels. Gradient channels remain an optional ablation.
# This grid is 4 augmentations x 3 losses x 2 samplers x 2 input feature modes = 48 runs.

IMAGE_TRANSFORM_EXPERIMENTS = [
    "none",
    "horizontal_flip",
    "brightness_contrast",
    "small_translation",
]

MASK_REPRESENTATION_EXPERIMENTS = [
    "binary_four_channel",  # fixed primary target
]

IMBALANCE_EXPERIMENTS = [
    "uniform",   # no reweighting / ordinary shuffle
    "category", # moderate image-level category-aware sampling
]

LOSS_EXPERIMENTS = [
    "bce",
    "bce_dice",
    "focal",
]

FEATURE_EXPERIMENTS = [
    "rgb_only",
    "gradient_features",
]

EXPERIMENTS = [
    {
        "transform": transform,
        "loss": loss_name,
        "sampler": sampler,
        "feature": feature,
        "mask_representation": "binary_four_channel",
    }
    for transform, loss_name, sampler, feature in itertools.product(
        IMAGE_TRANSFORM_EXPERIMENTS,
        LOSS_EXPERIMENTS,
        IMBALANCE_EXPERIMENTS,
        FEATURE_EXPERIMENTS,
    )
]


MAX_EXPERIMENTS=10
if MAX_EXPERIMENTS is not None:
    EXPERIMENTS = EXPERIMENTS[:MAX_EXPERIMENTS]

print("Experiments to run:", len(EXPERIMENTS))
display(pd.DataFrame(EXPERIMENTS).head(10))

# wandb.login() is usually unnecessary if already logged in, but safely ensures
# credentials are available in the notebook environment.
wandb.login()

experiment_results = []
for i, config in enumerate(EXPERIMENTS, start=1):
    result = train_experiment(config, i, len(EXPERIMENTS))
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results).sort_values(
    "test_score_proxy", ascending=False
).reset_index(drop=True)

results_path = "segmentation_experiment_comparison.csv"
results_df.to_csv(results_path, index=False)
display(results_df)
print("Saved:", results_path)

In [ ]:

# FINAL CELL — GENERATE submission.csv USING THE BEST VALIDATION CHECKPOINT

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
from torchvision.transforms import functional as TF

# ------------------------------------------------------------
# 1. Select the best experiment using validation loss
# ------------------------------------------------------------
# Do not choose a model using the held-out test score.
# Checkpoint files were saved as: <experiment_name>_best.pt

valid_results = results_df.dropna(subset=["best_val_loss"]).copy()

if valid_results.empty:
    raise ValueError("No completed experiments with valid checkpoints found.")

best_row = valid_results.sort_values("best_val_loss").iloc[0]
best_experiment = str(best_row["experiment"])
best_checkpoint = Path(f"{best_experiment}_best.pt")

if not best_checkpoint.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {best_checkpoint.resolve()}. "
        "Make sure the experiment output files still exist."
    )

best_config = {
    "transform": best_row["transform"],
    "loss": best_row["loss"],
    "sampler": best_row["sampler"],
    "feature": best_row["feature"],
}

print("Selected experiment:", best_experiment)
print("Best validation loss:", best_row["best_val_loss"])
print("Checkpoint:", best_checkpoint)

# ------------------------------------------------------------
# 2. Load the saved model
# ------------------------------------------------------------
model = build_model(
    feature_mode=best_config["feature"],
    pretrained=USE_PRETRAINED
).to(device)

checkpoint = torch.load(
    best_checkpoint,
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint)
model.eval()

# ------------------------------------------------------------
# 3. RLE encoder — column-major order, matching the decoder
# ------------------------------------------------------------
def encode_rle(mask):
    """
    mask: 2D binary array [H, W].
    Returns an empty string for an empty mask.
    Uses 1-indexed runs and Fortran/column-major order.
    """
    mask = np.asarray(mask, dtype=np.uint8)
    pixels = mask.flatten(order="F")

    padded = np.concatenate(([0], pixels, [0]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])

    starts = changes[::2] + 1
    lengths = changes[1::2] - changes[::2]

    if len(starts) == 0:
        return ""

    return " ".join(
        f"{start} {length}"
        for start, length in zip(starts, lengths)
    )

# ------------------------------------------------------------
# 4. Prepare competition test-image paths
# ------------------------------------------------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

test_image_paths = {
    p.stem: p
    for p in Path(TEST_IMG_DIR).rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
}

submission = sample_df.copy()
submission["image_id"] = submission["image_id"].astype(str)
submission["defect_type"] = submission["defect_type"].astype(str)

missing_ids = set(submission["image_id"]) - set(test_image_paths)
if missing_ids:
    raise FileNotFoundError(
        f"{len(missing_ids)} template image IDs have no matching test file. "
        f"Examples: {list(missing_ids)[:5]}"
    )

# ------------------------------------------------------------
# 5. Predict all four masks per image
# ------------------------------------------------------------
mean = IMAGENET_MEAN.to(device)
std = IMAGENET_STD.to(device)

def preprocess_test_image(image_path):
    with Image.open(image_path) as im:
        arr = np.asarray(im.convert("RGB"), dtype=np.uint8).copy()

    image = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0

    # Match the training preprocessing: native resolution + ImageNet norm.
    image = (image - IMAGENET_MEAN) / IMAGENET_STD
    return image.unsqueeze(0).to(device), arr.shape[:2]

# Each image is inferred only once, even if the template has four rows/image.
unique_test_ids = submission["image_id"].drop_duplicates().tolist()
predicted_rles = {}

with torch.inference_mode():
    for index, image_id in enumerate(unique_test_ids, start=1):
        image_tensor, (height, width) = preprocess_test_image(
            test_image_paths[image_id]
        )

        model_input = prepare_model_input(
            image_tensor, best_config["feature"]
        )

        logits = model(model_input)["out"]

        if logits.shape[-2:] != (height, width):
            logits = F.interpolate(
                logits,
                size=(height, width),
                mode="bilinear",
                align_corners=False
            )

        probabilities = torch.sigmoid(logits)[0].cpu().numpy()

        for category_index, category in enumerate(DEFECTS):
            binary_mask = (
                probabilities[category_index] >= THRESHOLD
            ).astype(np.uint8)

            predicted_rles[(image_id, category)] = encode_rle(binary_mask)

        if index % 100 == 0 or index == len(unique_test_ids):
            print(
                f"Predicted {index}/{len(unique_test_ids)} test images",
                flush=True
            )

# ------------------------------------------------------------
# 6. Fill the sample submission template in its original order
# ------------------------------------------------------------
# Detect the RLE column rather than assuming its exact name.
rle_candidates = [
    col for col in submission.columns
    if col.lower() in {"mask_rle", "rle", "encoded_pixels"}
]

if len(rle_candidates) != 1:
    raise ValueError(
        f"Expected exactly one RLE column in sample_df; "
        f"found {rle_candidates}. Columns: {submission.columns.tolist()}"
    )

rle_col = rle_candidates[0]

submission[rle_col] = [
    predicted_rles[(image_id, category)]
    for image_id, category in zip(
        submission["image_id"], submission["defect_type"]
    )
]

output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

# ------------------------------------------------------------
# 7. Validate the output
# ------------------------------------------------------------
saved_submission = pd.read_csv(output_path).fillna("")

assert len(saved_submission) == len(sample_df)
assert saved_submission.columns.tolist() == sample_df.columns.tolist()
assert not saved_submission[rle_col].isna().any()

print("\nSubmission saved:", output_path)
print("Rows:", len(saved_submission))
print("Columns:", saved_submission.columns.tolist())
print("Non-empty predictions:", int(
    saved_submission[rle_col].astype(str).str.strip().ne("").sum()
))
print(saved_submission.head())
